# ViT-equivalent optional geometry definitions

Purpose: define a candidate that contains the actual ViT exactly at initialization. Inputs: masked image tokens and original patch positions. Outputs: full JEPA online/EMA encoders, unchanged predictor/objective, frozen teacher operator diagnostics. Definitions import in seconds; Run All here does not train. The front door is `notebooks/15_ViT_Equivalent_Geometry.ipynb`; checkpoints and exact resume belong to its source-bound runner under `checkpoints/<run_id>/`.

Raw independent Q/K retain the ViT head dimension. Signed spatial bias can favor near or distant pairs and is **not a manifold metric**. Zero coefficient has a nonzero derivative. The default experiment trains this coefficient only; independently parameterized semantic temperature and spatial length remain frozen. No FMM acceleration is implemented in this notebook.

In [ ]:
from pathlib import Path
import json,os
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
_boot={'__file__':str(ROOT/'fmm_jepa/bootstrap.ipynb')}
for _cell in json.loads((ROOT/'fmm_jepa/bootstrap.ipynb').read_text())['cells']:
    if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):exec(compile(''.join(_cell['source']),'bootstrap#ahr-export','exec'),_boot)
_boot['install_notebook_package'](ROOT)


In [ ]:
"""ViT-containing directed pair bias. Definitions never load data or train."""
from pathlib import Path
import copy, math
import torch
from torch import nn
from torch.nn import functional as F
from fmm_jepa.models import JEPA, VisionEncoder, AttentionMixer, Predictor, patch_coordinates, position_features, validate_positions
BACKEND='exact-sdpa-vit-boundary-optional-signed-spatial-bias-v1'
VARIANTS=('vit','optional_geometry')

def project_qkv(mixer,x,valid):
    if x.ndim!=3 or x.shape[-1]!=mixer.dim or valid.shape!=x.shape[:2] or valid.dtype!=torch.bool:
        raise ValueError('Expected x[B,N,D] and boolean validity[B,N]')
    if not bool(valid.any(-1).all()):raise ValueError('Each sample needs a valid source')
    safe=torch.where(valid[...,None],x,torch.zeros_like(x))
    projected=mixer.qkv(safe).reshape(x.shape[0],x.shape[1],3,mixer.heads,mixer.dim//mixer.heads).permute(2,0,3,1,4)
    return projected.unbind(0)

class OptionalGeometryMixer(AttentionMixer):
    """Exact ViT at multiplier0/coefficient0; signed bias is NOT a manifold metric.
    Query and key remain actual independent raw ViT projections. Temperature and
    spatial length are independent. Default screen trains only the signed bias.
    """
    def __init__(self,dim,heads=3,spatial_coefficient_bound=2.,spatial_length=.35,
                 temperature_trainable=False,spatial_length_trainable=False,spatial_coefficient_trainable=True):
        super().__init__(dim,heads)
        bound=float(spatial_coefficient_bound);length=float(spatial_length)
        if not math.isfinite(bound) or not 0<bound<=8:raise ValueError('Signed coefficient bound must be in(0,8]')
        if not math.isfinite(length) or not .03<=length<=2:raise ValueError('Spatial length must be in[.03,2]')
        self.spatial_coefficient_bound=bound
        self.log_temperature_multiplier=nn.Parameter(torch.zeros(heads),requires_grad=bool(temperature_trainable))
        self.spatial_coefficient_raw=nn.Parameter(torch.zeros(heads),requires_grad=bool(spatial_coefficient_trainable))
        self.log_spatial_length=nn.Parameter(torch.full((heads,),math.log(length)),requires_grad=bool(spatial_length_trainable))
        self.backend=BACKEND
    def geometry_parameters(self,dtype=None):
        dtype=dtype or self.qkv.weight.dtype
        multiplier=self.log_temperature_multiplier.to(dtype).clamp(-math.log(16.),math.log(16.)).exp()
        coefficient=self.spatial_coefficient_bound*self.spatial_coefficient_raw.to(dtype).tanh()
        length=self.log_spatial_length.to(dtype).clamp(math.log(.03),math.log(2.)).exp()
        return {'temperature':math.sqrt(self.dim//self.heads)*multiplier,'temperature_multiplier':multiplier,
                'spatial_coefficient':coefficient,'spatial_length':length}
    def spatial_bias(self,indices,valid,grid,dtype):
        if indices.shape!=valid.shape or indices.dtype!=torch.long:raise ValueError('Matching original token indices required')
        positions=patch_coordinates(indices.clamp(0,grid[0]*grid[1]-1),grid,dtype=dtype)
        distance=(positions[:,:,None]-positions[:,None,:]).square().sum(-1)
        parameters=self.geometry_parameters(dtype)
        bias=-parameters['spatial_coefficient'][None,:,None,None]*distance[:,None]/(2*parameters['spatial_length'][None,:,None,None].square())
        return bias.masked_fill(~valid[:,None,None,:],float('-inf'))
    def kernel_state(self,x,indices,valid,grid):
        q,k,v=project_qkv(self,x,valid);p=self.geometry_parameters(q.dtype)
        logits=(q@k.transpose(-1,-2))/p['temperature'][None,:,None,None]+self.spatial_bias(indices,valid,grid,q.dtype)
        weights=logits.softmax(-1)
        weights=torch.where(valid[:,None,:,None],weights,torch.zeros_like(weights))
        return {'query':q,'key':k,'value':v,'weights':weights,'logits':logits,**p,
                'geometry':'Raw independent ViT Q/K plus optional bounded signed spatial pair bias; no metric/isomorphism claim'}
    def forward(self,x,indices=None,valid=None,grid=None):
        if indices is None or valid is None or grid is None:raise ValueError('Original indices, validity and grid required')
        q,k,v=project_qkv(self,x,valid)
        # SDPA scale is its normal sqrt(head_dim); a zero log-multiplier scales q by EXACT1.
        q=q*self.log_temperature_multiplier.to(q.dtype).clamp(-math.log(16.),math.log(16.)).neg().exp()[None,:,None,None]
        bias=self.spatial_bias(indices,valid,grid,q.dtype)
        output=F.scaled_dot_product_attention(q,k,v,attn_mask=bias,dropout_p=0.)
        output=output.transpose(1,2).reshape_as(x)
        return self.projection(output)*valid[...,None]

class OptionalGeometryEncoder(VisionEncoder):
    def __init__(self,patch_size=8,dim=192,depth=4,heads=3,**options):
        super().__init__(kind='vit',patch_size=patch_size,dim=dim,depth=depth,heads=heads)
        self.kind='optional_geometry'
        for block in self.blocks:block.mixer=OptionalGeometryMixer(dim,heads,**options)
    def embed(self,images,indices=None,valid=None):
        if images.ndim!=4 or images.shape[1]!=3:raise ValueError('Images[B,3,H,W] required')
        batch,_,height,width=images.shape;patch=self.patch_size
        if height%patch or width%patch:raise ValueError('Image dimensions must divide patch size')
        grid=(height//patch,width//patch);count=grid[0]*grid[1]
        if indices is None:
            if valid is not None:raise ValueError('Validity cannot be provided without indices')
            indices=torch.arange(count,device=images.device).expand(batch,-1)
        if valid is None:valid=torch.ones_like(indices,dtype=torch.bool)
        if indices.device!=images.device or valid.device!=images.device:raise ValueError('Input devices differ')
        validate_positions(indices,valid,batch,count)
        raw=F.unfold(images,kernel_size=patch,stride=patch).transpose(1,2)
        selected=raw.gather(1,indices.clamp(0,count-1)[...,None].expand(-1,-1,raw.shape[-1]))
        x=self.patch_embed(torch.where(valid[...,None],selected,torch.zeros_like(selected)))
        x=(x+position_features(indices.clamp(0,count-1),grid,self.dim,x.dtype))*valid[...,None]
        return x,indices,valid,grid

class OptionalGeometryJEPA(JEPA):
    def __init__(self,config):
        nn.Module.__init__(self);self.config=copy.deepcopy(config)
        options=copy.deepcopy(config.get('kernel15',{}));variant=options.pop('variant','optional_geometry')
        if variant!='optional_geometry':raise ValueError('Optional geometry configuration required')
        allowed={'spatial_coefficient_bound','spatial_length','temperature_trainable','spatial_length_trainable','spatial_coefficient_trainable'}
        if set(options)-allowed:raise ValueError('Unknown kernel15 options:'+str(sorted(set(options)-allowed)))
        self.context=OptionalGeometryEncoder(patch_size=config.get('patch_size',8),dim=config.get('dim',192),
            depth=config.get('depth',4),heads=config.get('heads',3),**options)
        self.target=copy.deepcopy(self.context).requires_grad_(False).eval()
        self.predictor=Predictor(self.context.dim,dim=config.get('predictor_dim',128),depth=config.get('predictor_depth',2),heads=config.get('predictor_heads',4))

def build_model(config,kind='semantic',variant=None):
    variant=variant or config.get('kernel15',{}).get('variant','optional_geometry')
    if variant not in VARIANTS:raise ValueError('Explicit registered kernel15 variant required')
    if kind=='vit' or variant=='vit':
        cfg=copy.deepcopy(config);cfg['kind']='vit';return JEPA(cfg)
    if kind!='semantic':raise ValueError('kind must be vit or semantic')
    return OptionalGeometryJEPA(config)

@torch.no_grad()
def initialize_from_vit(destination,reference):
    """Copy every actual ViT context/predictor tensor, retaining separate storage."""
    source=reference.context.state_dict();target=destination.context.state_dict();copied=[]
    for name,value in source.items():
        if name not in target or target[name].shape!=value.shape:raise ValueError('Actual ViT tensor missing:'+name)
        target[name].copy_(value);copied.append('context.'+name)
    destination.context.load_state_dict(target);destination.predictor.load_state_dict(reference.predictor.state_dict())
    copied+=['predictor.'+name for name in reference.predictor.state_dict()]
    for name in copied:
        a=reference.state_dict()[name];b=destination.state_dict()[name]
        if not torch.equal(a,b) or a.data_ptr()==b.data_ptr():raise AssertionError('Exact shared values/separate storage failed:'+name)
    for block in destination.context.blocks:
        if hasattr(block.mixer,'spatial_coefficient_raw'):
            block.mixer.spatial_coefficient_raw.zero_();block.mixer.log_temperature_multiplier.zero_()
    destination.target.load_state_dict(destination.context.state_dict());destination.target.requires_grad_(False).eval()
    return copied

def parameter_counts(model):
    count=lambda module:sum(p.numel() for p in module.parameters())
    trainable=lambda module:sum(p.numel() for p in module.parameters() if p.requires_grad)
    return {'variant':model.context.kind,'encoder_total':count(model.context),'encoder_active':trainable(model.context),
            'encoder_inactive_scaffold':0,'predictor':count(model.predictor),'trainable_total':trainable(model),
            'trainable_active':trainable(model),'with_ema_teacher_total':count(model),'head_dimension':model.context.dim//model.context.blocks[0].mixer.heads,
            'backend':BACKEND,'geometry':'Optional signed spatial bias; not a proved manifold metric','fmm_acceleration':False}

@torch.no_grad()
def kernel_geometry_diagnostics(model_or_encoder,images,indices=None,valid=None,max_samples=2,**options):
    encoder=getattr(model_or_encoder,'context',model_or_encoder)
    if encoder.kind=='vit':return {'status':'NOT_APPLICABLE','reason':'Ordinary ViT; reference raw Q/K measured by frozen_attention_states'}
    prior=encoder.training;encoder.eval();layers=[]
    try:
        x,indices,valid,grid=encoder.embed(images[:max_samples],indices[:max_samples]if indices is not None else None,valid[:max_samples]if valid is not None else None)
        for index,block in enumerate(encoder.blocks):
            state=block.mixer.kernel_state(block.norm1(x),indices,valid,grid);w=state['weights'];mask=valid[:,None].expand(-1,block.mixer.heads,-1)
            entropy=-(w*w.clamp_min(torch.finfo(w.dtype).tiny).log()).sum(-1)[mask]
            layers.append({'block':index,'head_dimension':block.mixer.dim//block.mixer.heads,'temperature':state['temperature'].cpu().tolist(),
                'spatial_coefficient':state['spatial_coefficient'].cpu().tolist(),'spatial_length':state['spatial_length'].cpu().tolist(),
                'kernel_entropy_mean':float(entropy.mean()),'effective_neighbors_exp_entropy_mean':float(entropy.exp().mean()),
                'geometry':state['geometry'],'dense_only':True})
            x=block(x,indices,valid,grid)
        return {'status':'MEASURED','variant':encoder.kind,'grid':list(grid),'samples':len(images[:max_samples]),'layers':layers,
                'interpretation':'Descriptive optional signed bias. No manifold isomorphism or FMM acceleration claim.'}
    finally:encoder.train(prior)

@torch.no_grad()
def frozen_attention_states(encoder,images,indices=None,valid=None,max_samples=4,blocks=None,layer_index=None):
    """Actual frozen teacher normalized tokens and raw Q/K/V; no labels or fitting."""
    encoder=getattr(encoder,'context',encoder)
    if layer_index is not None:blocks=[int(layer_index)]
    if not hasattr(encoder,'blocks'):raise ValueError('Vision encoder required')
    images=images[:max_samples];batch,_,height,width=images.shape;grid=(height//encoder.patch_size,width//encoder.patch_size);count=grid[0]*grid[1]
    if indices is None:indices=torch.arange(count,device=images.device).expand(batch,-1)
    else:indices=indices[:max_samples]
    if valid is None:valid=torch.ones_like(indices,dtype=torch.bool)
    else:valid=valid[:max_samples]
    validate_positions(indices,valid,batch,count)
    raw=F.unfold(images,kernel_size=encoder.patch_size,stride=encoder.patch_size).transpose(1,2)
    selected=raw.gather(1,indices.clamp(0,count-1)[...,None].expand(-1,-1,raw.shape[-1]))
    x=encoder.patch_embed(torch.where(valid[...,None],selected,torch.zeros_like(selected)))
    x=(x+position_features(indices.clamp(0,count-1),grid,encoder.dim,x.dtype))*valid[...,None]
    prior=encoder.training;encoder.eval();records=[]
    try:
        for index,block in enumerate(encoder.blocks):
            normalized=block.norm1(x)
            if blocks is None or index in blocks:
                q,k,v=project_qkv(block.mixer,normalized,valid)
                exact=F.scaled_dot_product_attention(q,k,v,attn_mask=valid[:,None,None,:],dropout_p=0.)*valid[:,None,:,None]
                records.append({'block':index,'tokens':normalized.detach().clone(),'query':q.detach().clone(),'key':k.detach().clone(),
                    'value':v.detach().clone(),'indices':indices.detach().clone(),'valid':valid.detach().clone(),'grid':grid,
                    'positions':patch_coordinates(indices.clamp(0,count-1),grid,dtype=q.dtype),'reference_head_output':exact.detach().clone(),
                    'reference_role':'Exact actual ViT Q/K/V dot-product attention on frozen teacher token states'})
            x=block(x,indices,valid,grid)
        if layer_index is not None:
            if len(records)!=1:raise ValueError('Requested layer absent')
            record=records[0];record.update(q=record['query'],k=record['key'],v=record['value'],raw_teacher_output=record['reference_head_output'])
            return record
        return records
    finally:encoder.train(prior)

@torch.no_grad()
def constraint_attention(record,constraint='reference',spatial_coefficient=.25,spatial_length=.35,content_scale=.5,projection=None):
    """One constraint at a time on the SAME teacher Q/K/V. No trained model arm.
    Compression uses one shared training-only PCA projection for Q and K; the
    reference sqrt(head_dim) scale is retained to isolate coordinate truncation.
    """
    q,k,v=record['query'],record['key'],record['value'];temperature=math.sqrt(q.shape[-1]);valid=record['valid']
    if constraint=='bounded':q=content_scale*q.tanh();k=content_scale*k.tanh()
    elif constraint in ('compressed6','compressed32'):
        dimension=6 if constraint=='compressed6' else 32
        if projection is None or projection.shape!=(q.shape[1],q.shape[-1],dimension):raise ValueError('Frozen train-only projection[H,D,d] required')
        q=torch.einsum('bhnd,hde->bhne',q,projection);k=torch.einsum('bhnd,hde->bhne',k,projection)
    elif constraint not in ('reference','spatial'):raise ValueError('Unknown isolated constraint')
    logits=(q@k.transpose(-1,-2))/temperature
    if constraint=='spatial':
        positions=record['positions'];distance=(positions[:,:,None]-positions[:,None,:]).square().sum(-1)
        logits=logits-float(spatial_coefficient)*distance[:,None]/(2*float(spatial_length)**2)
    weights=logits.masked_fill(~valid[:,None,None,:],float('-inf')).softmax(-1)
    return (weights@v)*valid[:,None,:,None]

@torch.no_grad()
def fit_shared_projection(records,dimension):
    """Uncentered PCA of training-only raw Q/K; shared per head, frozen after fitting."""
    if not records or dimension<1:raise ValueError('Nonempty training records and positive dimension required')
    heads=records[0]['query'].shape[1];width=records[0]['query'].shape[-1]
    if dimension>width:raise ValueError('Projection dimension exceeds head dimension')
    result=[]
    for head in range(heads):
        rows=torch.cat([tensor[:,head][record['valid']].double().cpu() for record in records for tensor in (record['query'],record['key'])])
        eigenvalues,eigenvectors=torch.linalg.eigh(rows.T@rows)
        vectors=eigenvectors[:,-dimension:].flip(-1)
        pivots=vectors.abs().argmax(0);signs=vectors[pivots,torch.arange(dimension)].sign();vectors=vectors*signs
        result.append(vectors)
    return torch.stack(result).to(device=records[0]['query'].device,dtype=records[0]['query'].dtype)


@torch.no_grad()
def diagnostic_geometry(q,k,v,valid,positions,spatial_coefficient=.25,spatial_length=.35):
    """Six isolated constraints on frozen teacher states; no labels/model selection.
    Truncation is a genuine deterministic first-coordinate slice, NOT an optimal
    low-rank fit and NOT proof that all d-dimensional routing must have this error.
    """
    temperature=math.sqrt(q.shape[-1]);mask=valid[:,None,:,None].expand(*q.shape[:-1],v.shape[-1])
    base_logits=(q@k.transpose(-1,-2))/temperature
    def output(logits):
        weights=logits.masked_fill(~valid[:,None,None,:],float('-inf')).softmax(-1)
        return weights,(weights@v)*valid[:,None,:,None]
    reference_weights,reference=output(base_logits);denominator=reference[mask].double().norm().clamp_min(1e-15)
    cases=[('reference',base_logits,'Actual raw ViT Q/K and standard temperature'),
        ('spatial_only',base_logits-float(spatial_coefficient)*(positions[:,:,None]-positions[:,None,:]).square().sum(-1)[:,None]/(2*float(spatial_length)**2),'Only signed spatial pair bias changed'),
        ('temperature_only',base_logits/2,'Only semantic temperature doubled'),
        ('bounded_only',(.5*q.tanh())@(.5*k.tanh()).transpose(-1,-2)/temperature,'Only raw Q/K bounded by0.5*tanh')]
    for dimension in (6,32):
        if dimension<=q.shape[-1]:cases.append((f'truncated{dimension}',q[...,:dimension]@k[...,:dimension].transpose(-1,-2)/temperature,'Deterministic first-coordinate slice; original sqrt(head_dim) temperature retained'))
    rows=[]
    rowmask=valid[:,None].expand(-1,q.shape[1],-1)
    for name,logits,scope in cases:
        weights,prediction=output(logits);difference=(prediction-reference)[mask].double()
        entropy=-(weights*weights.clamp_min(torch.finfo(weights.dtype).tiny).log()).sum(-1)[rowmask]
        rows.append({'constraint':name,'relative_output_l2_error':float(difference.norm()/denominator),
            'output_rmse':float(difference.square().mean().sqrt()),'attention_row_l1_error_mean':float((weights-reference_weights).abs().sum(-1)[rowmask].mean()),
            'kernel_entropy_nats_mean':float(entropy.mean()),'samples':q.shape[0],'heads':q.shape[1],'tokens':q.shape[2],
            'head_dimension':q.shape[-1],'scope':scope,'evidence_role':'FROZEN_TRAIN_ONLY_OPERATOR_DIAGNOSTIC','fmm_acceleration':False})
    return rows

@torch.no_grad()
def geometry_parameter_summary(model_or_encoder):
    encoder=getattr(model_or_encoder,'context',model_or_encoder);rows=[]
    for index,block in enumerate(encoder.blocks):
        mixer=block.mixer
        if not hasattr(mixer,'geometry_parameters'):continue
        parameters=mixer.geometry_parameters()
        rows.append({'block':index,**{k:v.detach().cpu().tolist() for k,v in parameters.items()},
            'signed_bias_not_metric':True,'temperature_trainable':mixer.log_temperature_multiplier.requires_grad,
            'spatial_length_trainable':mixer.log_spatial_length.requires_grad,'spatial_coefficient_trainable':mixer.spatial_coefficient_raw.requires_grad,
            'spatial_coefficient_gradient_norm':None if mixer.spatial_coefficient_raw.grad is None else float(mixer.spatial_coefficient_raw.grad.norm())})
    return rows
